# Object Detection with Open Images v7 and Webcam

In this notebook we prepare an object detection dataset from **Open Images v7** using **FiftyOne**, then train a YOLO model to detect everyday objects on a live webcam feed.

**Steps:**
1. Import libraries
2. Choose target classes
3. Download the dataset subset
4. Inspect and export to YOLO format *(next)*
5. Train and run real-time detection *(next)*

In [1]:
import fiftyone as fo
import fiftyone.zoo as foz

In [2]:
print(fo.list_datasets())

You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
[]


## 1. Target classes

We pick **96 everyday classes** that are likely to appear in front of a webcam:

- **Electronics and office:** phone, laptop, keyboard, mouse, monitor, calculator, stapler, ...
- **Kitchen and tableware:** mug, plate, bowl, spoon, fork, knife, teapot, kettle, ...
- **Food and drinks:** fruits, vegetables, fast food, bakery items, sweets, beverages
- **Personal items and home decor:** watch, sunglasses, hat, handbag, lamp, vase, pillow, ...
- **People and animals:** person, human face, human hand, cat, dog

Class names must match the Open Images names exactly (capitalized as in the official list). The next cell prints the number of classes as a sanity check.

In [4]:
labels = ["Mobile phone", "Laptop", "Camera", "Computer keyboard", "Computer mouse", "Headphones",
          "Mug", "Bottle", "Coffee cup", "Book", "Scissors", "Glasses", "Pen", "Clock", "Backpack",
          "Person", "Chair",
          "Grape", "Strawberry", "Pear", "Peach", "Pineapple", "Watermelon", "Mango",
          "Banana", "Apple", "Orange", "Lemon", "Tomato", "Carrot", "Cucumber", "Potato",
          "Pizza", "Hamburger", "Hot dog", "French fries", "Sandwich", "Fast food", "Popcorn", "Salad",
          "Bread", "Bagel", "Pretzel", "Muffin", "Croissant", "Cake", "Cookie", "Candy", "Ice cream",
          "Milk", "Juice", "Tea", "Beer", "Wine", "Cocktail","Tablet computer", "Computer monitor", "Television", "Remote control", "Microphone",
          "Calculator", "Ruler", "Stapler", "Eraser", "Pencil case", "Envelope", "Whiteboard",
          "Plate", "Bowl", "Spoon", "Fork", "Knife", "Teapot", "Kettle", "Wine glass", "Tin can",
          "Watch", "Sunglasses", "Hat", "Handbag", "Umbrella", "Toothbrush", "Towel",
          "Lamp", "Picture frame", "Houseplant", "Vase", "Pillow", "Candle", "Mirror",
          "Teddy bear", "Cat", "Dog", "Dumbbell", "Human face", "Human hand"]
print(len(labels))

96


## 2. Download the dataset

We download the **train** split of Open Images v7 with bounding-box labels (`detections`).

- `classes`: only images containing at least one of the target classes
- `max_samples=50000`: total number of images (not per class)
- `shuffle=True`: random selection instead of the first matches
- `dataset_name`: name of the dataset registered in FiftyOne

> The first run downloads large metadata files (about 4.8 GB), so it can take several minutes. Later runs use the local cache in `~/fiftyone/open-images-v7`.

In [ ]:
train_ds = foz.load_zoo_dataset(
    "open-images-v7",
    classes = labels,
    split= "train",
    dataset_name= "train_dataset",
    max_samples= 50000, 
    shuffle= True, 
    label_types= ['detections'],
    persistent= True
)

Found 242 images, downloading the remaining 49758


## 3. Download the validation split

We download the **validation** split of Open Images v7 using the same target classes and the same label type (`detections`), so the model can be evaluated on images it has never seen during training.

- `split="validation"`: the validation part of Open Images v7
- `classes=labels`: the same list of target classes used for the train split
- `max_samples=10000`: total number of images (not per class), about 20% of the training size
- `shuffle=True`: random selection instead of the first matches
- `dataset_name="validation_dataset"`: name of the dataset registered in FiftyOne
- `label_types=["detections"]`: bounding boxes for object detection

> The train and validation splits are separate in Open Images, so there is no overlap between them. The validation metadata is much smaller than the train metadata, so this step is faster than the previous download.

In [ ]:
validation_ds = foz.load_zoo_dataset(
    "open-images-v7",
    classes = labels,
    split= "validation",
    dataset_name= "validation_dataset",
    max_samples= 10000, 
    shuffle= True, 
    label_types= ['detections'], 
    persistent= True
)